# MARIS 09 — ResNet50V2 Attention U-Net with Focal Tversky Loss

In [1]:
import os, sys, json, time, random, zipfile, shutil
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import tensorflow as tf
from tensorflow.keras import layers, Model, callbacks

# -------------------------------------------------------------------------
# Kaggle paths: inputs are read-only; all generated files go to /kaggle/working.
# Attach the MARIS dataset to the notebook using "Add Input".
# -------------------------------------------------------------------------
IS_KAGGLE = Path("/kaggle/input").exists()

if IS_KAGGLE:
    INPUT_DIR = Path("/kaggle/input")
    WORK_DIR = Path("/kaggle/working")
    print("[*] Kaggle environment detected")

    split_candidates = sorted(INPUT_DIR.rglob("dataset_splits.csv"))
    if not split_candidates:
        raise FileNotFoundError(
            "dataset_splits.csv was not found under /kaggle/input. "
            "Use Kaggle's Add Input to attach the dataset containing "
            "ml/processed/dataset_splits.csv."
        )
    SPLITS_CSV = split_candidates[0]
    # Expected project layout: <project>/ml/processed/dataset_splits.csv
    ROOT = SPLITS_CSV.parent.parent.parent
else:
    print("[*] Local environment detected")
    here = Path.cwd().resolve()
    split_candidates = list(here.glob("**/ml/processed/dataset_splits.csv"))
    if not split_candidates:
        split_candidates = list(here.glob("**/dataset_splits.csv"))
    if not split_candidates:
        raise FileNotFoundError("Could not find ml/processed/dataset_splits.csv")
    SPLITS_CSV = split_candidates[0]
    ROOT = SPLITS_CSV.parent.parent.parent

MODELS_DIR = WORK_DIR / "models" if IS_KAGGLE else ROOT / "ml" / "models"
RESULTS_DIR = WORK_DIR / "results" if IS_KAGGLE else ROOT / "ml" / "results"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def resolve_data_path(value):
    """Resolve paths stored in dataset_splits.csv across common Kaggle layouts."""
    p = Path(str(value))
    candidates = []
    if p.is_absolute():
        candidates.append(p)
    else:
        candidates.extend([ROOT / p, ROOT / "dataset" / p, SPLITS_CSV.parent / p])
        if IS_KAGGLE:
            candidates.extend([Path("/kaggle/input") / p])
    for candidate in candidates:
        if candidate.exists():
            return candidate
    # Last-resort basename search for datasets whose CSV paths include an old local prefix.
    if IS_KAGGLE:
        matches = list(Path("/kaggle/input").rglob(p.name))
        if len(matches) == 1:
            return matches[0]
    raise FileNotFoundError(f"Could not resolve dataset file: {value}")

print(f"[✓] Project root       : {ROOT}")
print(f"[✓] Splits CSV         : {SPLITS_CSV}")
print(f"[✓] Models output      : {MODELS_DIR}")
print(f"[✓] Results output     : {RESULTS_DIR}")
print(f"[✓] TensorFlow         : {tf.__version__}")
gpus = tf.config.list_physical_devices("GPU")
print(f"[✓] GPUs               : {gpus if gpus else 'None detected (CPU mode)'}")

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


[*] Kaggle environment detected
[✓] Project root       : /kaggle/input/datasets/navjotsingh20/maris-sar-dataset
[✓] Splits CSV         : /kaggle/input/datasets/navjotsingh20/maris-sar-dataset/ml/processed/dataset_splits.csv
[✓] Models output      : /kaggle/working/models
[✓] Results output     : /kaggle/working/results
[✓] TensorFlow         : 2.21.0
[✓] GPUs               : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


## 1. Focal Tversky Loss & Segmentation Metrics
- $\beta = 0.5$
- $\alpha = 0.5$
- $\gamma = 0.75$

In [2]:
@tf.keras.utils.register_keras_serializable()
def focal_tversky_loss(y_true, y_pred, alpha=0.5, beta=0.5, gamma=0.75, smooth=1e-6):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    tp = tf.reduce_sum(y_true * y_pred)
    fp = tf.reduce_sum((1.0 - y_true) * y_pred)
    fn = tf.reduce_sum(y_true * (1.0 - y_pred))
    tversky = (tp + smooth) / (tp + alpha * fp + beta * fn + smooth)
    return tf.pow(1.0 - tversky, gamma)

@tf.keras.utils.register_keras_serializable()
def dice_score(y_true, y_pred, smooth=1e-6):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred > 0.5, tf.float32)
    intersection = tf.reduce_sum(y_true * y_pred)
    return (2.0 * intersection + smooth) / (tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) + smooth)

@tf.keras.utils.register_keras_serializable()
def iou_score(y_true, y_pred, smooth=1e-6):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred > 0.5, tf.float32)
    intersection = tf.reduce_sum(y_true * y_pred)
    union = tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) - intersection
    return (intersection + smooth) / (union + smooth)

## 2. Model Architecture: ResNet50V2 Attention U-Net

Connects multi-stage skip taps from the pre-trained ResNet50V2 backbone through Additive Attention Gates to the upsampling decoder.

In [3]:
def attention_gate(skip, gate, filters):
    """Additive Attention Gate to suppress background sea clutter."""
    a = layers.Conv2D(filters, 1, padding='same')(skip)
    b = layers.Conv2D(filters, 1, padding='same')(gate)
    x = layers.Activation('relu')(layers.Add()([a, b]))
    alpha = layers.Conv2D(1, 1, activation='sigmoid')(x)
    return layers.Multiply()([skip, alpha])

def decoder_block(x, skip, filters):
    """Upsampling block with Attention Gate skip filtering."""
    x = layers.UpSampling2D((2, 2))(x)
    s = attention_gate(skip, x, filters)
    x = layers.Concatenate()([x, s])
    x = layers.Conv2D(filters, 3, padding='same', activation='relu')(x)
    x = layers.Conv2D(filters, 3, padding='same', activation='relu')(x)
    return x

def build_resnet_attention_unet(input_shape=(256, 256, 3)):
    base_model = tf.keras.applications.ResNet50V2(
        include_top=False,
        # Set to None if internet is disabled and ImageNet weights are not cached.
        weights='imagenet',
        input_shape=input_shape
    )
    base_model.trainable = True

    # Encoder skip connection tapping
    s1 = base_model.get_layer('conv1_conv').output        # (128, 128, 64)
    s2 = base_model.get_layer('conv2_block2_out').output   # (64, 64, 256)
    s3 = base_model.get_layer('conv3_block3_out').output   # (32, 32, 512)
    s4 = base_model.get_layer('conv4_block5_out').output   # (16, 16, 1024)
    b  = base_model.get_layer('conv5_block3_out').output   # (8, 8, 2048) Bottleneck

    d4 = decoder_block(b, s4, 512)
    d3 = decoder_block(d4, s3, 256)
    d2 = decoder_block(d3, s2, 128)
    d1 = decoder_block(d2, s1, 64)

    u0 = layers.UpSampling2D((2, 2))(d1)
    c0 = layers.Conv2D(32, 3, padding='same', activation='relu')(u0)
    outputs = layers.Conv2D(1, 1, activation='sigmoid', name='slick_mask')(c0)

    return Model(inputs=base_model.input, outputs=outputs, name='ResNet50V2_Attention_UNet')

model = build_resnet_attention_unet()
print(f'[✓] Model Name: {model.name}')
print(f'[✓] Total Parameters: {model.count_params():,}')
print(f'[✓] Input: {model.input_shape} -> Output: {model.output_shape}')

94668760/94668760 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
[✓] Model Name: ResNet50V2_Attention_UNet
[✓] Total Parameters: 45,841,669
[✓] Input: (None, 256, 256, 3) -> Output: (None, 256, 256, 1)


## 3. Data Generator with Standardization


In [4]:
IMG_SIZE = (256, 256)
MEAN = np.array([0.4243585, 0.4243585, 0.4243585], dtype=np.float32)
STD  = np.array([0.2139608, 0.2139608, 0.2139608], dtype=np.float32)
BATCH_SIZE = 8

class SARDatasetGenerator(tf.keras.utils.Sequence):
    def __init__(self, df, batch_size=BATCH_SIZE, shuffle=True):
        self.df = df.reset_index(drop=True)
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indices = np.arange(len(self.df))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.df) / self.batch_size))

    def __getitem__(self, idx):
        batch_idx = self.indices[idx * self.batch_size : (idx + 1) * self.batch_size]
        batch_images, batch_masks = [], []

        for i in batch_idx:
            row = self.df.iloc[i]
            img = Image.open(resolve_data_path(row['image_path'])).convert('RGB').resize(IMG_SIZE)
            img_arr = (np.array(img, dtype=np.float32) / 255.0 - MEAN) / STD

            mask = Image.open(resolve_data_path(row['mask_path'])).convert('L').resize(IMG_SIZE, Image.Resampling.NEAREST)
            mask_arr = (np.array(mask) > 127).astype(np.float32)

            batch_images.append(img_arr)
            batch_masks.append(np.expand_dims(mask_arr, axis=-1))

        return np.array(batch_images), np.array(batch_masks)

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

df = pd.read_csv(SPLITS_CSV)
train_df = df[df['split'] == 'train']
val_df   = df[df['split'] == 'val']
test_df  = df[df['split'] == 'test']

print(f'[✓] Dataset Splits: {len(train_df):,} Train | {len(val_df):,} Val | {len(test_df):,} Test')
train_gen = SARDatasetGenerator(train_df, batch_size=BATCH_SIZE, shuffle=True)
val_gen   = SARDatasetGenerator(val_df, batch_size=BATCH_SIZE, shuffle=False)
test_gen  = SARDatasetGenerator(test_df, batch_size=BATCH_SIZE, shuffle=False)

[✓] Dataset Splits: 6,455 Train | 807 Val | 808 Test


## 4. Model Compilation & Training

In [5]:
LEARNING_RATE = 1e-4
EPOCHS = 15

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss=focal_tversky_loss,
    metrics=[dice_score, iou_score]
)

SAVE_PATH = MODELS_DIR / 'best_resnet_attention_unet.keras'

cb_list = [
    callbacks.ModelCheckpoint(
        str(SAVE_PATH),
        monitor='val_dice_score',
        mode='max',
        save_best_only=True,
        verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    ),
    callbacks.EarlyStopping(
        monitor='val_dice_score',
        mode='max',
        patience=5,
        restore_best_weights=True,
        verbose=1
    )
]

print(f'Ready to train ResNet50V2 Attention U-Net for {EPOCHS} epochs.')
history = model.fit(train_gen, validation_data=val_gen, epochs=EPOCHS, callbacks=cb_list)

Ready to train ResNet50V2 Attention U-Net for 15 epochs.


/usr/local/lib/python3.13/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/15


E0000 00:00:1795598001.261417      71 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1795598008.041309      71 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1795598020.269124      71 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1795598021.211425      71 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1795598022.414251      71 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


734/807 ━━━━━━━━━━━━━━━━━━━━ 1:31 1s/step - dice_score: 0.7561 - iou_score: 0.6234 - loss: 0.3447

E0000 00:00:1795598986.235827      70 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1795599007.550947      70 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1795599008.143756      70 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1795599010.109960      70 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1795599013.276422      70 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000

807/807 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - dice_score: 0.7600 - iou_score: 0.6281 - loss: 0.3404
Epoch 1: val_dice_score improved from None to 0.82843, saving model to /kaggle/working/models/best_resnet_attention_unet.keras

Epoch 1: finished saving model to /kaggle/working/models/best_resnet_attention_unet.keras
807/807 ━━━━━━━━━━━━━━━━━━━━ 1319s 2s/step - dice_score: 0.8000 - iou_score: 0.6757 - loss: 0.2955 - val_dice_score: 0.8284 - val_iou_score: 0.7141 - val_loss: 0.2624 - learning_rate: 1.0000e-04
Epoch 2/15
807/807 ━━━━━━━━━━━━━━━━━━━━ 0s 314ms/step - dice_score: 0.8307 - iou_score: 0.7161 - loss: 0.2603
Epoch 2: val_dice_score improved from 0.82843 to 0.84430, saving model to /kaggle/working/models/best_resnet_attention_unet.keras

Epoch 2: finished saving model to /kaggle/working/models/best_resnet_attention_unet.keras
807/807 ━━━━━━━━━━━━━━━━━━━━ 268s 331ms/step - dice_score: 0.8336 - iou_score: 0.7202 - loss: 0.2569 - val_dice_score: 0.8443 - val_iou_score: 0.7377 - val_loss

## 5. Test Set Benchmark & Evaluation

In [6]:
best_model_path = SAVE_PATH if SAVE_PATH.exists() else None
if best_model_path and best_model_path.exists():
    print('[*] Loading checkpoint:', best_model_path)
    eval_model = tf.keras.models.load_model(
        best_model_path,
        custom_objects={
            'focal_tversky_loss': focal_tversky_loss,
            'dice_score': dice_score,
            'iou_score': iou_score
        }
    )
else:
    print('[*] Evaluating current trained weights.')
    eval_model = model

all_preds, all_masks, all_probabilities = [], [], []
for i in range(len(test_gen)):
    x_b, y_b = test_gen[i]
    preds = eval_model.predict(x_b, verbose=0)
    all_probabilities.append(preds.astype(np.float32))
    all_preds.append((preds > 0.5).astype(np.uint8))
    all_masks.append(y_b.astype(np.uint8))

flat_preds = np.concatenate(all_preds).reshape(-1)
flat_masks = np.concatenate(all_masks).reshape(-1)
flat_probabilities = np.concatenate(all_probabilities).reshape(-1)

# Compute the same Focal Tversky loss used during training over the full test set.
test_loss = focal_tversky_loss(
    tf.convert_to_tensor(flat_masks, dtype=tf.float32),
    tf.convert_to_tensor(flat_probabilities, dtype=tf.float32)
).numpy()

intersection = np.logical_and(flat_masks, flat_preds).sum()
union = np.logical_or(flat_masks, flat_preds).sum()
dice = (2.0 * intersection) / (flat_masks.sum() + flat_preds.sum() + 1e-7)
iou = intersection / (union + 1e-7)
precision = intersection / (flat_preds.sum() + 1e-7)
recall = intersection / (flat_masks.sum() + 1e-7)

results = {
    'Model': 'ResNet50V2 Attention U-Net (Focal Tversky)',
    'Loss': round(float(test_loss), 4),
    'Dice': round(float(dice), 4),
    'IoU': round(float(iou), 4),
    'Precision': round(float(precision), 4),
    'Recall': round(float(recall), 4)
}
df_res = pd.DataFrame([results])
print(f'\nBenchmark Performance on {len(test_df)} Test Images:')
print(df_res.to_string(index=False))
df_res.to_csv(RESULTS_DIR / 'resnet_attention_unet_results.csv', index=False)

# Save training history when available.
if "history" in globals() and hasattr(history, "history"):
    pd.DataFrame(history.history).to_csv(RESULTS_DIR / "training_history.csv", index=False)

# Save evaluation metrics in machine-readable form.
with open(RESULTS_DIR / "evaluation_metrics.json", "w") as f:
    json.dump(results, f, indent=2)


[*] Loading checkpoint: /kaggle/working/models/best_resnet_attention_unet.keras

Benchmark Performance on 808 Test Images:
                                     Model   Loss   Dice    IoU  Precision  Recall
ResNet50V2 Attention U-Net (Focal Tversky) 0.2067 0.8778 0.7822     0.8643  0.8918


## 6. Qualitative Visualizations on Test SAR Images

In [7]:
# Visualize 3 test predictions
fig, axes = plt.subplots(max(1, len(sample_indices)), 4, figsize=(16, 4 * max(1, len(sample_indices))))
if len(sample_indices) == 1:
    axes = np.expand_dims(axes, axis=0)
sample_indices = sorted(set([0, min(50, len(test_df)-1), min(120, len(test_df)-1)])) if len(test_df) else []

for row_idx, s_idx in enumerate(sample_indices):
    sample_row = test_df.iloc[s_idx]
    raw_img = Image.open(resolve_data_path(sample_row['image_path'])).convert('RGB').resize(IMG_SIZE)
    raw_mask = Image.open(resolve_data_path(sample_row['mask_path'])).convert('L').resize(IMG_SIZE, Image.Resampling.NEAREST)
    
    img_arr = (np.array(raw_img, dtype=np.float32) / 255.0 - MEAN) / STD
    pred = eval_model.predict(np.expand_dims(img_arr, 0), verbose=0)[0, :, :, 0]
    pred_bin = (pred > 0.5).astype(np.uint8)
    
    axes[row_idx, 0].imshow(raw_img)
    axes[row_idx, 0].set_title(f'Sample {s_idx}: Input SAR', fontsize=10)
    axes[row_idx, 0].axis('off')
    
    axes[row_idx, 1].imshow(raw_mask, cmap='gray')
    axes[row_idx, 1].set_title('Ground Truth Mask', fontsize=10)
    axes[row_idx, 1].axis('off')
    
    axes[row_idx, 2].imshow(pred, cmap='magma')
    axes[row_idx, 2].set_title('Raw Sigmoid Probability', fontsize=10)
    axes[row_idx, 2].axis('off')
    
    axes[row_idx, 3].imshow(raw_img)
    axes[row_idx, 3].imshow(pred_bin, cmap='autumn', alpha=0.45)
    axes[row_idx, 3].set_title('Predicted Oil Slick Overlay', fontsize=10)
    axes[row_idx, 3].axis('off')

if sample_indices:
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'resnet_sample_predictions.png', dpi=180, bbox_inches='tight')
    plt.show()
else:
    print('No test samples available for visualization.')

NameError: name 'sample_indices' is not defined

## 7. Package All Outputs for Download

This cell collects the trained model, evaluation metrics, training history, and visualizations into a single ZIP archive. In Kaggle, the archive is saved under `/kaggle/working/results/` and is available from the notebook's **Output** panel after the run completes.

In [ ]:
# Ensure a final model file exists even if no best-checkpoint was written.
final_model_path = MODELS_DIR / "resnet_attention_unet_final.keras"
if "model" in globals() and not final_model_path.exists():
    model.save(final_model_path)

# Keep the best checkpoint in the package when it exists.
archive_path = WORK_DIR / "MARIS_09_all_results.zip" if IS_KAGGLE else ROOT / "MARIS_09_all_results.zip"
archive_path.parent.mkdir(parents=True, exist_ok=True)

# Remove an archive from a previous run before packaging, so the new ZIP
# never contains an older copy of itself.
results_zip = RESULTS_DIR / "MARIS_09_all_results.zip"
if results_zip.exists():
    results_zip.unlink()

# Package both output directories. The ZIP is created outside RESULTS_DIR first,
# then copied into RESULTS_DIR so it does not try to include itself.
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for folder in (MODELS_DIR, RESULTS_DIR):
        if folder.exists():
            for file_path in folder.rglob("*"):
                if file_path.is_file():
                    zf.write(file_path, arcname=str(file_path.relative_to(WORK_DIR if IS_KAGGLE else ROOT)))

# Also place a copy inside results/ as requested.
if archive_path.resolve() != results_zip.resolve():
    shutil.copy2(archive_path, results_zip)

print(f"[✓] ZIP archive created: {results_zip}")
print("[✓] Included output files:")
for file_path in sorted(RESULTS_DIR.rglob("*")):
    if file_path.is_file():
        print(" -", file_path.relative_to(RESULTS_DIR))
